# face_cropper — usage demo

This notebook shows how to use the **same face detector the production model service uses** to crop faces in your training pipeline. Three patterns:

1. **As a library** — `from face_cropper import crop_face` in any notebook cell
2. **As a CLI from inside the notebook** — `!python face_cropper.py crop-dir …`
3. **Batch from Python** — `crop_directory(in_dir, out_dir)` for in-process preprocessing

> **Canonical source:** `application/model_service/core/face_detector.py` (re-exported via `face_cropper.py` at the repo root). If you tune the detector, do it there — this demo and the CLI inherit changes automatically.

## 1. Setup

If you're running this notebook from inside the repo (`face_cropper/demo.ipynb`), the import below works out of the box. If you copied it into a Kaggle/Colab environment, make sure `face_cropper.py` and `application/model_service/core/face_detector.py` are reachable on `sys.path`.

In [ ]:
import sys
from pathlib import Path

# This notebook lives at face_cropper/demo.ipynb, so the repo root is one level up.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'face_cropper' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from face_cropper import crop_face, crop_directory, FaceDetector

print('repo root:', REPO_ROOT)
print('FaceDetector source module:', FaceDetector.__module__)
print('  ↑ confirms we are using the same class the model service uses')

## 2. Pick a sample image

Edit `SAMPLE_IMAGE` to point at any face image you have locally. The cell below also offers a default that works inside this repo.

In [ ]:
# Edit me ↓
SAMPLE_IMAGE = REPO_ROOT / 'dataset' / 'processed' / 'sfew' / '0' / '000026.jpg'

if not SAMPLE_IMAGE.exists():
    print(f'\u26a0  {SAMPLE_IMAGE} not found — set SAMPLE_IMAGE to any face image you have.')
else:
    print(f'using {SAMPLE_IMAGE}')

## 3. Library — single image

`crop_face(image)` accepts a path string, a `pathlib.Path`, a `PIL.Image`, or a uint8 BGR numpy array. By default it returns a `PIL.Image` (or `None` if no face was detected).

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

face = crop_face(SAMPLE_IMAGE)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(Image.open(SAMPLE_IMAGE))
ax[0].set_title('input')
ax[0].axis('off')
if face is not None:
    ax[1].imshow(face)
    ax[1].set_title(f'crop {face.size}')
else:
    ax[1].set_title('no face detected')
ax[1].axis('off')
plt.tight_layout()
plt.show()

## 4. Other input types

Same function — different things you can pass in. Useful when you've already loaded the image (e.g. via your data loader) and don't want to round-trip through disk.

In [ ]:
import cv2
import numpy as np

# numpy (BGR — what cv2.imread gives you)
bgr = cv2.imread(str(SAMPLE_IMAGE), cv2.IMREAD_COLOR)
face_np = crop_face(bgr, return_pil=False)
print(f'numpy → numpy : shape={face_np.shape}, dtype={face_np.dtype}')

# PIL.Image
pil = Image.open(SAMPLE_IMAGE)
face_pil = crop_face(pil)
print(f'PIL   → PIL   : size={face_pil.size}, mode={face_pil.mode}')

## 5. Padding

A tight face crop loses jaw, hairline, and shoulder context. For training an emotion classifier, a small fractional padding (~0.1) often improves accuracy — worth experimenting per-dataset. The model service uses **no padding** at inference, so keep training/inference consistent if you depend on the model service's pipeline.

In [ ]:
tight   = crop_face(SAMPLE_IMAGE, padding=0.0)
padded  = crop_face(SAMPLE_IMAGE, padding=0.15)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(tight);  ax[0].set_title(f'padding=0.0  {tight.size}');  ax[0].axis('off')
ax[1].imshow(padded); ax[1].set_title(f'padding=0.15 {padded.size}'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 6. Batch from Python — `crop_directory`

Walks an input directory, crops every image, writes results to the output directory preserving the relative subpath. Returns a summary dict you can inspect right away.

In [ ]:
import tempfile, shutil

# Build a tiny temp dataset so the demo doesn't depend on your dataset layout.
demo_in  = Path(tempfile.mkdtemp(prefix='facecrop_in_'))
demo_out = Path(tempfile.mkdtemp(prefix='facecrop_out_'))
for i in range(3):
    shutil.copy(SAMPLE_IMAGE, demo_in / f'sample_{i}.jpg')

summary = crop_directory(
    demo_in,
    demo_out,
    recursive=True,
    padding=0.1,
    resize=224,        # square 224×224 — matches EmpathBot input
    skip_existing=False,
)

print('summary:', summary)
print('outputs:', sorted(p.name for p in demo_out.iterdir()))

## 7. Batch from the CLI — same thing, no Python required

If you prefer the command line (or want to script preprocessing outside a notebook), the CLI does the same work. From inside this notebook we just shell out with `!`.

In [ ]:
cli_out = Path(tempfile.mkdtemp(prefix='facecrop_cli_'))
report  = cli_out / 'report.json'

!python {REPO_ROOT}/face_cropper.py crop-dir {demo_in} {cli_out} \
    --recursive --resize 224 --padding 0.1 \
    --skip-existing --report {report}

## 8. Inspect the report

The `--report` JSON is the most useful dataset-quality signal. `no_face` is the count of images where the detector found nothing — those are worth eyeballing before you train, since they'll never contribute to the loss.

In [ ]:
import json
report_data = json.loads(report.read_text())
print(json.dumps(report_data, indent=2))

yield_rate = report_data['cropped'] / max(report_data['total'], 1)
print(f'\nyield: {yield_rate:.1%} of input images produced a crop')

## 9. Reference

- **Canonical detector**: `application/model_service/core/face_detector.py`
- **CLI / library wrapper**: `face_cropper.py` (repo root)
- **Docs**: `face_cropper/README.md`
- **Smoke test**: `python face_cropper/test_face_cropper.py <optional face image>`

If you change detector behaviour (confidence threshold, weights, picking criterion), do it in `face_detector.py`. This notebook and the CLI will pick up the change next run — no copy-and-paste.